In [5]:
import torch
import open_clip
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')

c:\Learning\Labs\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Learning\Labs\venv\lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


In [22]:
caption_model = torch.load('caption_features_flickr8k.pt')
image_model = torch.load('image_features_flickr8k.pt')

In [37]:
image_names = list(image_model.keys())
image_embs = torch.stack(list(image_model.values()))  # shape: [N_img, 512]

In [ ]:
caption_embs = torch.stack([cap[2] for cap in caption_model])  # shape: [N_cap, 512]
caption_img_names = [cap[0] for cap in caption_model]


In [64]:
caption_embs = caption_embs.squeeze(1)
image_embs = image_embs.squeeze(1)

In [65]:
print("caption_embs shape:", caption_embs.shape)   # should be [num_captions, 512]
print("image_embs shape:", image_embs.shape)       # should be [num_images, 512]

print("caption_embs[0] shape:", caption_embs[0].shape)  # should be [512]
print("caption_embs[0].unsqueeze(0) shape:", caption_embs[0].unsqueeze(0).shape)  # [1,512]

caption_embs shape: torch.Size([40455, 512])
image_embs shape: torch.Size([8091, 512])
caption_embs[0] shape: torch.Size([512])
caption_embs[0].unsqueeze(0) shape: torch.Size([1, 512])


In [ ]:
# Text to image
# Image to text
# text to text
# image to image

import torch.nn.functional as F

def text_to_image_recall(image_embs, caption_embs, caption_img_names, image_names, K=10):
    recalls = 0
    total = len(caption_embs)


    for i in range(total):
        sims = F.cosine_similarity(caption_embs[i].unsqueeze(0), image_embs).squeeze()
        topk = sims.topk(10).indices
        
        retrieved_imgs = [image_names[j] for j in topk]
        true_img = caption_img_names[i]

        if true_img in retrieved_imgs:
            recalls += 1

    print(f"Recall@{10}: {recalls/total:.4f}")
    return recalls / total

10
Recall@10: 0.0000


In [49]:
text_to_image_recall(image_embs, caption_embs, caption_img_names, image_names, K=5)

8091


TypeError: only integer tensors of a single element can be converted to an index